# CTR Prediction — Initial EDA

Exploring a sample of the Avazu CTR dataset. Uses `data_loader.load_sample()` — see `../consts.py` for the sample-size and data-path config.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from consts import CONTEXT_FEATURE_COLS, LABEL_COL
from data_loader import load_sample, validate_schema
from feature_engineering import add_hour_features

pd.set_option("display.max_columns", 50)

In [ ]:
df = load_sample(n_rows=500_000)
validate_schema(df)
df = add_hour_features(df)
df.shape

## Shape, dtypes, nulls

In [ ]:
df.dtypes

In [ ]:
df.isnull().mean().sort_values(ascending=False)

## Click distribution / class imbalance

Avazu's overall CTR is known to be roughly 17% — sanity-check the sample against that.

In [ ]:
ctr = df[LABEL_COL].mean()
print(f"overall CTR: {ctr:.4f}")
df[LABEL_COL].value_counts(normalize=True).plot(kind="bar", title="click distribution")
plt.show()

## Feature distributions split by click

A handful of candidate user/ad/context columns, colored by click rate.

In [ ]:
import seaborn as sns

candidate_cols = ["device_type", "site_category", "app_category", "banner_pos", "hour_of_day"]
fig, axes = plt.subplots(len(candidate_cols), 1, figsize=(8, 4 * len(candidate_cols)))
for ax, col in zip(axes, candidate_cols):
    df.groupby(col)[LABEL_COL].mean().plot(kind="bar", ax=ax, title=f"CTR by {col}")
plt.tight_layout()
plt.show()

## Univariate signal check

Mean click rate per category for each candidate feature — an early read on
predictive power before modeling.

In [ ]:
for col in candidate_cols + CONTEXT_FEATURE_COLS:
    print(col)
    print(df.groupby(col)[LABEL_COL].mean().sort_values(ascending=False).head(5))
    print()

## Note: device_id / device_ip cardinality

Known Avazu quirk — a small number of "default"/shared `device_id` values
dominate the dataset (many devices report the same placeholder ID), so
`device_id` alone is a weak user proxy. Check cardinality and the top values'
share before relying on it heavily.

In [ ]:
for col in ["device_id", "device_ip"]:
    vc = df[col].value_counts(normalize=True)
    print(f"{col}: {df[col].nunique()} unique values, top value covers {vc.iloc[0]:.2%} of rows")